In [ ]:
# Load packages
import os
import hail as hl
import pandas as pd 

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
        "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

In [ ]:
# Run this cell once
hl.init(
    gcs_requester_pays_configuration=os.environ['GOOGLE_CLOUD_PROJECT']
)

In [ ]:
# Donwload exclusion samples 
!gcloud storage cp gs://vwb-aou-datasets-controlled/v8/known_issues/wgs_v8_known_issue_1.txt . --billing-project=${GOOGLE_CLOUD_PROJECT}

In [ ]:
# Exclusion samples 
exclusion = pd.read_csv('wgs_v8_known_issue_1.txt', sep='\t')
samples_to_exclude = exclusion['research_id'].values

In [ ]:
# ============================================================
# Load MatrixTable
# ============================================================

mt_wgs_path = (
    "gs://vwb-aou-datasets-controlled/"
    "v8/wgs/short_read/snpindel/exome/splitMT/hail.mt"
)

mt = hl.read_matrix_table(mt_wgs_path)


# ============================================================
# Prepare exclusion samples ONCE
# ============================================================

samples_to_exclude = {
    str(x) for x in samples_to_exclude
}

print(f"Samples to exclude: {len(samples_to_exclude)}")


# ============================================================
# Regions
# ============================================================

regions = [
    "chr2:25230000-25320000",
    "chr4:105140000-105280000",
    "chr5:177510000-177520000",
    "chr15:90080000-90110000",
    "chr17:76730000-76740000"
]

# ============================================================
# Process regions
# ============================================================

for region_string in regions:

    print(f"\n{'=' * 70}")
    print(f"Processing {region_string}")
    print(f"{'=' * 70}")

    # --------------------------------------------------------
    # Define genomic interval
    # --------------------------------------------------------

    interval = hl.parse_locus_interval(
        region_string,
        reference_genome="GRCh38"
    )

    # --------------------------------------------------------
    # Filter MatrixTable to region
    # --------------------------------------------------------

    mt_region = hl.filter_intervals(
        mt,
        [interval]
    )

    # --------------------------------------------------------
    # Remove excluded samples
    # --------------------------------------------------------

    mt_region = mt_region.filter_cols(
        ~hl.literal(samples_to_exclude).contains(
            mt_region.s
        )
    )

    # --------------------------------------------------------
    # Convert to entry table
    # --------------------------------------------------------

    ht = mt_region.entries()

    # --------------------------------------------------------
    # Keep only called ALT genotypes
    # --------------------------------------------------------

    ht = ht.filter(
        hl.is_defined(ht.GT) &
        (ht.GT.n_alt_alleles() > 0)
    )

    # --------------------------------------------------------
    # Calculate DP
    # --------------------------------------------------------

    ht = ht.annotate(
        DP=hl.if_else(
            hl.is_defined(ht.AD),
            hl.sum(ht.AD),
            hl.missing(hl.tint32)
        )
    )

    # --------------------------------------------------------
    # Calculate VAF and AB
    # --------------------------------------------------------

    ht = ht.annotate(
        VAF=hl.if_else(
            hl.is_defined(ht.AD) &
            (ht.DP > 0),
            ht.AD[1] / ht.DP,
            hl.missing(hl.tfloat64)
        ),

        AB=hl.if_else(
            hl.is_defined(ht.AD) &
            (ht.DP > 0) &
            ht.GT.is_het(),
            ht.AD[1] / ht.DP,
            hl.missing(hl.tfloat64)
        )
    )

    # --------------------------------------------------------
    # Remove locus/s columns from table key
    #
    # entries() creates a key on locus + s.
    # This allows us to select locus and s as ordinary fields.
    # --------------------------------------------------------

    ht = ht.key_by()

    # --------------------------------------------------------
    # Keep required fields
    # --------------------------------------------------------

    ht = ht.select(
        "locus",
        "alleles",
        "s",
        "DP",
        "VAF",
        "AB",
        "GT",
        "AD",
        "GQ",
        "PS",
        "RGQ",
        "FT"
    )

    # --------------------------------------------------------
    # Increase partitions before export
    # --------------------------------------------------------

    print("Repartitioning entry table...")

    ht = ht.repartition(
        100,
        shuffle=True
    )

    # --------------------------------------------------------
    # Output filename
    # --------------------------------------------------------

    region_name = (
        region_string
        .replace(":", "_")
        .replace("-", "_")
    )

    output = (
        f'{os.environ["WORKSPACE_BUCKET"]}/data/hail/'
        f"variant_genotype_information_"
        f"{region_name}.tsv.bgz"
    )

    # --------------------------------------------------------
    # Export
    # --------------------------------------------------------

    print(f"Exporting {output}...")

    ht.export(output)

    print(f"Finished {output}")